<a href="https://colab.research.google.com/github/alexacoonline/7006SCN_CAC_17089427/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Check if Java is installed and determine JAVA_HOME
java_exec_path = os.popen('readlink -f $(which java)').read().strip()

if java_exec_path and os.path.exists(java_exec_path):
    # JAVA_HOME should point to the directory containing the 'bin' folder
    java_home_path = os.path.dirname(os.path.dirname(java_exec_path))
    os.environ["JAVA_HOME"] = java_home_path
    print(f"Java is installed. JAVA_HOME set to: {os.environ['JAVA_HOME']}")
else:
    print("Java not found. Please ensure Java 11 or higher is installed and in your PATH.")
    print("You might need to install it with: !apt-get install openjdk-17-jdk-headless -qq")

!java -version
print("Current JAVA_HOME is set to:", os.environ.get('JAVA_HOME'))

Java is installed. JAVA_HOME set to: /usr/lib/jvm/java-17-openjdk-amd64
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
Current JAVA_HOME is set to: /usr/lib/jvm/java-17-openjdk-amd64


In [ ]:
import warnings
warnings.filterwarnings('ignore') # Filter out the Hugging Face warnings

!pip install pyspark datasets pyarrow

from pyspark.sql import SparkSession

# Stop any existing Spark session to ensure a clean restart
if 'spark' in locals() and spark.sparkContext._jsc is not None:
    print("Stopping existing Spark session...")
    spark.stop()

spark = SparkSession.builder \
    .appName("FineWeb_BigData") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


from datasets import load_dataset
import pyarrow as pa
import pyarrow.parquet as pq

dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    split="train",
    streaming=True
)


CHUNK_SIZE = 200000   # write every 200k rows
MAX_ROWS = 1000000    # 1M sample from dataset
buffer = []
file_index = 0
count = 0

for row in dataset:
    buffer.append(row)
    count += 1

    if len(buffer) >= CHUNK_SIZE:
        table = pa.Table.from_pylist(buffer)
        pq.write_table(table, f"fineweb_chunk_{file_index}.parquet")
        buffer = []
        file_index += 1
        print(f"Saved chunk {file_index}")

    if count >= MAX_ROWS:
        break

# write remaining
if buffer:
    table = pa.Table.from_pylist(buffer)
    pq.write_table(table, f"fineweb_chunk_{file_index}.parquet")

print("Streaming + storage complete")


df = spark.read.parquet("fineweb_chunk_*.parquet")

df.printSchema()
df.show(5)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

### Check and Install Java (if necessary)

PySpark requires Java to run. Let's check if Java is already installed and install OpenJDK 11 if it's not present.

In [ ]:
from pyspark.sql.functions import *

df = df.dropna(subset=["text"])
df = df.filter(length(col("text")) > 100)
df = df.dropDuplicates(["text"])

print("After cleaning:", df.count())


df = df.withColumn("text_length", length(col("text")))
df = df.withColumn("word_count", size(split(col("text"), " ")))
df = df.withColumn("avg_word_length", col("text_length") / col("word_count"))


df = df.withColumn(
    "label",
    when((col("text_length") > 2000) & (col("word_count") > 300), 1).otherwise(0)
)

df.groupBy("label").count().show()

from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF

tokenizer = Tokenizer(inputCol="text", outputCol="words")
df = tokenizer.transform(df)

remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
df = remover.transform(df)

hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=10000)
df = hashingTF.transform(df)

idf = IDF(inputCol="rawFeatures", outputCol="features")
idf_model = idf.fit(df)
df = idf_model.transform(df)
